In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

In [ ]:
import zipfile
import os

zip_file_path = '/content/archive.zip'
extract_dir = '/content/extracted_data'

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Archive extracted to: {extract_dir}")

print("Contents of the extracted directory:")
for root, dirs, files in os.walk(extract_dir):
    for name in files:
        print(os.path.join(root, name))
    for name in dirs:
        print(os.path.join(root, name))

Archive extracted to: /content/extracted_data
Contents of the extracted directory:
/content/extracted_data/german_credit_data.csv


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Unnamed: 0        1000 non-null   int64 
 1   Age               1000 non-null   int64 
 2   Sex               1000 non-null   object
 3   Job               1000 non-null   int64 
 4   Housing           1000 non-null   object
 5   Saving accounts   817 non-null    object
 6   Checking account  606 non-null    object
 7   Credit amount     1000 non-null   int64 
 8   Duration          1000 non-null   int64 
 9   Purpose           1000 non-null   object
dtypes: int64(5), object(5)
memory usage: 78.3+ KB


In [ ]:
data.describe(include='all')

,Unnamed: 0,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose
count,1000.000000,1000.000000,1000,1000.000000,1000,817,606,1000.000000,1000.000000,1000
unique,NaN,NaN,2,NaN,3,4,3,NaN,NaN,8
top,NaN,NaN,male,NaN,own,little,little,NaN,NaN,car
freq,NaN,NaN,690,NaN,713,603,274,NaN,NaN,337
mean,499.500000,35.546000,NaN,1.904000,NaN,NaN,NaN,3271.258000,20.903000,NaN
std,288.819436,11.375469,NaN,0.653614,NaN,NaN,NaN,2822.736876,12.058814,NaN
min,0.000000,19.000000,NaN,0.000000,NaN,NaN,NaN,250.000000,4.000000,NaN
25%,249.750000,27.000000,NaN,2.000000,NaN,NaN,NaN,1365.500000,12.000000,NaN
50%,499.500000,33.000000,NaN,2.000000,NaN,NaN,NaN,2319.500000,18.000000,NaN
75%,749.250000,42.000000,NaN,2.000000,NaN,NaN,NaN,3972.250000,24.000000,NaN


In [ ]:

print("\nValue counts for 'Saving accounts':")
print(data['Saving accounts'].value_counts(dropna=False))

print("\nValue counts for 'Checking account':")
print(data['Checking account'].value_counts(dropna=False))


Value counts for 'Saving accounts':
Saving accounts
little        603
NaN           183
moderate      103
quite rich     63
rich           48
Name: count, dtype: int64

Value counts for 'Checking account':
Checking account
NaN         394
little      274
moderate    269
rich         63
Name: count, dtype: int64


In [ ]:
data = data.drop('Unnamed: 0', axis=1)
print("DataFrame after dropping 'Unnamed: 0' column:")
print(data.head())

DataFrame after dropping 'Unnamed: 0' column:
   Age     Sex  Job Housing Saving accounts Checking account  Credit amount  \
0   67    male    2     own             NaN           little           1169   
1   22  female    2     own          little         moderate           5951   
2   49    male    1     own          little              NaN           2096   
3   45    male    2    free          little           little           7882   
4   53    male    2    free          little           little           4870   

   Duration              Purpose  
0         6             radio/TV  
1        48             radio/TV  
2        12            education  
3        42  furniture/equipment  
4        24                  car  


In [ ]:
print("\nValue counts for 'Saving accounts':")
print(data['Saving accounts'].value_counts(dropna=False))

print("\nValue counts for 'Checking account':")
print(data['Checking account'].value_counts(dropna=False))


Value counts for 'Saving accounts':
Saving accounts
little        603
NaN           183
moderate      103
quite rich     63
rich           48
Name: count, dtype: int64

Value counts for 'Checking account':
Checking account
NaN         394
little      274
moderate    269
rich         63
Name: count, dtype: int64


In [ ]:
saving_accounts_mode = data['Saving accounts'].mode()[0]
data['Saving accounts'] = data['Saving accounts'].fillna(saving_accounts_mode)

checking_account_mode = data['Checking account'].mode()[0]
data['Checking account'] = data['Checking account'].fillna(checking_account_mode)

print("Missing values after imputation:")
print(data[['Saving accounts', 'Checking account']].isnull().sum())

Missing values after imputation:
Saving accounts     0
Checking account    0
dtype: int64


In [ ]:
categorical_cols = data.select_dtypes(include='object').columns

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    print(f"Column '{col}' encoded with mapping: {list(le.classes_)}")

print("\nDataFrame after Label Encoding:")
print(data.head())


DataFrame after Label Encoding:
   Age  Sex  Job  Housing  Saving accounts  Checking account  Credit amount  \
0   67    1    2        1                0                 0           1169   
1   22    0    2        1                0                 1           5951   
2   49    1    1        1                0                 0           2096   
3   45    1    2        0                0                 0           7882   
4   53    1    2        0                0                 0           4870   

   Duration  Purpose  
0         6        5  
1        48        5  
2        12        3  
3        42        4  
4        24        1  


In [ ]:
print("Unique values in 'Job' (potential target variable):")
print(data['Job'].unique())

X = data.drop('Job', axis=1)
y = data['Job']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"\nShape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Unique values in 'Job' (potential target variable):
[2 1 3 0]

Shape of X_train: (700, 8)
Shape of X_test: (300, 8)
Shape of y_train: (700,)
Shape of y_test: (300,)


In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("RandomForestClassifier trained successfully.")

RandomForestClassifier trained successfully.


In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(report)

Accuracy: 0.6233

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         7
           1       0.43      0.14      0.21        71
           2       0.65      0.91      0.76       189
           3       0.38      0.15      0.22        33

    accuracy                           0.62       300
   macro avg       0.37      0.30      0.30       300
weighted avg       0.56      0.62      0.55       300



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("RandomForestClassifier trained successfully.")

RandomForestClassifier trained successfully.


In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(report)

Accuracy: 0.6233

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         7
           1       0.43      0.14      0.21        71
           2       0.65      0.91      0.76       189
           3       0.38      0.15      0.22        33

    accuracy                           0.62       300
   macro avg       0.37      0.30      0.30       300
weighted avg       0.56      0.62      0.55       300



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("RandomForestClassifier trained successfully.")

RandomForestClassifier trained successfully.


In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(report)

Accuracy: 0.6233

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         7
           1       0.43      0.14      0.21        71
           2       0.65      0.91      0.76       189
           3       0.38      0.15      0.22        33

    accuracy                           0.62       300
   macro avg       0.37      0.30      0.30       300
weighted avg       0.56      0.62      0.55       300



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("RandomForestClassifier trained successfully.")

RandomForestClassifier trained successfully.


In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(report)

Accuracy: 0.6233

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         7
           1       0.43      0.14      0.21        71
           2       0.65      0.91      0.76       189
           3       0.38      0.15      0.22        33

    accuracy                           0.62       300
   macro avg       0.37      0.30      0.30       300
weighted avg       0.56      0.62      0.55       300



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("RandomForestClassifier trained successfully.")

RandomForestClassifier trained successfully.


In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(report)

Accuracy: 0.6233

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         7
           1       0.43      0.14      0.21        71
           2       0.65      0.91      0.76       189
           3       0.38      0.15      0.22        33

    accuracy                           0.62       300
   macro avg       0.37      0.30      0.30       300
weighted avg       0.56      0.62      0.55       300



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("RandomForestClassifier trained successfully.")

RandomForestClassifier trained successfully.


In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(report)

Accuracy: 0.6233

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         7
           1       0.43      0.14      0.21        71
           2       0.65      0.91      0.76       189
           3       0.38      0.15      0.22        33

    accuracy                           0.62       300
   macro avg       0.37      0.30      0.30       300
weighted avg       0.56      0.62      0.55       300



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("RandomForestClassifier trained successfully.")

RandomForestClassifier trained successfully.


In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(report)

Accuracy: 0.6233

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         7
           1       0.43      0.14      0.21        71
           2       0.65      0.91      0.76       189
           3       0.38      0.15      0.22        33

    accuracy                           0.62       300
   macro avg       0.37      0.30      0.30       300
weighted avg       0.56      0.62      0.55       300



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("RandomForestClassifier trained successfully.")

RandomForestClassifier trained successfully.


In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(report)

Accuracy: 0.6233

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         7
           1       0.43      0.14      0.21        71
           2       0.65      0.91      0.76       189
           3       0.38      0.15      0.22        33

    accuracy                           0.62       300
   macro avg       0.37      0.30      0.30       300
weighted avg       0.56      0.62      0.55       300



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
categorical_cols = data.select_dtypes(include='object').columns

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    print(f"Column '{col}' encoded with mapping: {list(le.classes_)}")

print("\nDataFrame after Label Encoding:")
print(data.head())


DataFrame after Label Encoding:
   Age  Sex  Job  Housing  Saving accounts  Checking account  Credit amount  \
0   67    1    2        1                0                 0           1169   
1   22    0    2        1                0                 1           5951   
2   49    1    1        1                0                 0           2096   
3   45    1    2        0                0                 0           7882   
4   53    1    2        0                0                 0           4870   

   Duration  Purpose  
0         6        5  
1        48        5  
2        12        3  
3        42        4  
4        24        1  


In [ ]:
print("Unique values in 'Job' (potential target variable):")
print(data['Job'].unique())

X = data.drop('Job', axis=1)
y = data['Job']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"\nShape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Unique values in 'Job' (potential target variable):
[2 1 3 0]

Shape of X_train: (700, 8)
Shape of X_test: (300, 8)
Shape of y_train: (700,)
Shape of y_test: (300,)


In [ ]:
categorical_cols = data.select_dtypes(include='object').columns

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    print(f"Column '{col}' encoded with mapping: {list(le.classes_)}")

print("\nDataFrame after Label Encoding:")
print(data.head())


DataFrame after Label Encoding:
   Age  Sex  Job  Housing  Saving accounts  Checking account  Credit amount  \
0   67    1    2        1                0                 0           1169   
1   22    0    2        1                0                 1           5951   
2   49    1    1        1                0                 0           2096   
3   45    1    2        0                0                 0           7882   
4   53    1    2        0                0                 0           4870   

   Duration  Purpose  
0         6        5  
1        48        5  
2        12        3  
3        42        4  
4        24        1  


In [ ]:
print("Unique values in 'Job' (potential target variable):")
print(data['Job'].unique())

X = data.drop('Job', axis=1)
y = data['Job']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"\nShape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Unique values in 'Job' (potential target variable):
[2 1 3 0]

Shape of X_train: (700, 8)
Shape of X_test: (300, 8)
Shape of y_train: (700,)
Shape of y_test: (300,)


In [ ]:
categorical_cols = data.select_dtypes(include='object').columns

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    print(f"Column '{col}' encoded with mapping: {list(le.classes_)}")

print("\nDataFrame after Label Encoding:")
print(data.head())

Column 'Sex' encoded with mapping: ['female', 'male']
Column 'Housing' encoded with mapping: ['free', 'own', 'rent']
Column 'Saving accounts' encoded with mapping: ['little', 'moderate', 'quite rich', 'rich']
Column 'Checking account' encoded with mapping: ['little', 'moderate', 'rich']
Column 'Purpose' encoded with mapping: ['business', 'car', 'domestic appliances', 'education', 'furniture/equipment', 'radio/TV', 'repairs', 'vacation/others']

DataFrame after Label Encoding:
   Age  Sex  Job  Housing  Saving accounts  Checking account  Credit amount  \
0   67    1    2        1                0                 0           1169   
1   22    0    2        1                0                 1           5951   
2   49    1    1        1                0                 0           2096   
3   45    1    2        0                0                 0           7882   
4   53    1    2        0                0                 0           4870   

   Duration  Purpose  
0         6        5 

In [ ]:
print("Unique values in 'Job' (potential target variable):")
print(data['Job'].unique())

X = data.drop('Job', axis=1)
y = data['Job']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"\nShape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Unique values in 'Job' (potential target variable):
[2 1 3 0]

Shape of X_train: (700, 8)
Shape of X_test: (300, 8)
Shape of y_train: (700,)
Shape of y_test: (300,)


In [ ]:
saving_accounts_mode = data['Saving accounts'].mode()[0]
data['Saving accounts'] = data['Saving accounts'].fillna(saving_accounts_mode)

checking_account_mode = data['Checking account'].mode()[0]
data['Checking account'] = data['Checking account'].fillna(checking_account_mode)

print("Missing values after imputation:")
print(data[['Saving accounts', 'Checking account']].isnull().sum())

Missing values after imputation:
Saving accounts     0
Checking account    0
dtype: int64


In [ ]:
saving_accounts_mode = data['Saving accounts'].mode()[0]
data['Saving accounts'] = data['Saving accounts'].fillna(saving_accounts_mode)

checking_account_mode = data['Checking account'].mode()[0]
data['Checking account'] = data['Checking account'].fillna(checking_account_mode)

print("Missing values after imputation:")
print(data[['Saving accounts', 'Checking account']].isnull().sum())

Missing values after imputation:
Saving accounts     0
Checking account    0
dtype: int64


In [ ]:
saving_accounts_mode = data['Saving accounts'].mode()[0]
data['Saving accounts'] = data['Saving accounts'].fillna(saving_accounts_mode)

checking_account_mode = data['Checking account'].mode()[0]
data['Checking account'] = data['Checking account'].fillna(checking_account_mode)

print("Missing values after imputation:")
print(data[['Saving accounts', 'Checking account']].isnull().sum())

Missing values after imputation:
Saving accounts     0
Checking account    0
dtype: int64
